In [1]:
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
from plot_progress import gather_metrics
from parse_levels import find_levels_in_configs, find_levels_in_configs_glob
from parse_levels import process_metrics, human_train_time_dict, convert_to_dict, compute_gap_in_percentage
from parse_levels import filter_folder_info
import pandas as pd
import matplotlib.ticker as ticker
from plot_utils import plot_gap_comparison
import re
import copy
import json

In [2]:
# for each item in the dict, if any two have the same 'record', remove the one with lower number of steps in metric
def deduplicate_metrics(search_results):
    records_length_so_far = {}
    new_search_results = {}
    for key, value in search_results.items():
        if value['record'] not in records_length_so_far:
            records_length_so_far[value['record']] = (len(value['metrics']['step']), key)
            new_search_results[key] = value
        else:
            if len(value['metrics']['step']) > records_length_so_far[value['record']][0]:
                new_search_results.pop(records_length_so_far[value['record']][1])

                records_length_so_far[value['record']] = (len(value['metrics']['step']), key)
                new_search_results[key] = value
    return new_search_results



In [3]:
# ori_results = find_levels_in_configs_glob(
#     [
#         '/checkpoint/maui/zhaobc/scientist/workspace/record_*',
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250424_*',
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250425_*',
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250412_*',
#         #  '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250408_*',
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_*_20250405_*'
#         # '/checkpoint/maui/zhaobc/scientist/workspace/record_1[2-8]_20250409_*',
#     ]
# )
with open('results.cache', 'r') as f:
    ori_results = json.load(f)

In [4]:
folder_info = ori_results

In [5]:
len(folder_info)

888

In [6]:
# flat search -- n_initial_hypotheses = 50
flat_params = [
    ('runner', 'bon'),
    ('n_initial_hypotheses', 50),
]
# tree search -- n_initial_hypotheses = 1, n_hypotheses = 3
tree_params = [
    ('runner', 'bon'),
    ('n_initial_hypotheses', 1),
    ('n_hypotheses', 3),
]
# forest search -- n_initial_hypotheses = 3, n_hypotheses = 3
forest_params = [
    ('runner', 'bon'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 3),
]
# AIDE -- n_initial_hypotheses = 3, n_hypotheses = 1, debug_prob = 0.5
aide_params = [
    ('runner', 'aide'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 1),
    ('debug_prob', 0.5),
]
# MultiAIDE -- n_initial_hypotheses = 3, n_hypotheses = 3, debug_prob = 0.5
multi_aide_params = [
    ('runner', 'aide'),
    ('n_initial_hypotheses', 3),
    ('n_hypotheses', 3),
    ('debug_prob', 0.5),
]
# for each search algo, we get the R1 and O3 with level 1, 12, 125 results for plotting
# thats 6 results for each search algo, so we plot with 3 columns (level) and 2 rows (R1 and O3)

search_algo_params = {
    'flat': flat_params,
    'tree': tree_params,
    'forest': forest_params,
    'aide': aide_params,
    'multi_aide': multi_aide_params,
}

plot_info = {}

for search_algo, params in search_algo_params.items():
    plot_info[search_algo] = {}
    for level in [1, 12, 125]:
        plot_info[search_algo][level] = {}
        for model in ['deepseek-r1', 'o3-mini']:
            search_params = params + [('levels', level), ('model', model)]
            filtered_folder_info = filter_folder_info(folder_info, search_params)
            print(f'{search_algo} {level} {model} {len(filtered_folder_info)}')
            plot_info[search_algo][level][model] = filtered_folder_info



flat 1 deepseek-r1 18
flat 1 o3-mini 18
flat 12 deepseek-r1 18
flat 12 o3-mini 18
flat 125 deepseek-r1 18
flat 125 o3-mini 18
tree 1 deepseek-r1 18
tree 1 o3-mini 18
tree 12 deepseek-r1 18
tree 12 o3-mini 18
tree 125 deepseek-r1 18
tree 125 o3-mini 18
forest 1 deepseek-r1 18
forest 1 o3-mini 18
forest 12 deepseek-r1 18
forest 12 o3-mini 18
forest 125 deepseek-r1 36
forest 125 o3-mini 36
aide 1 deepseek-r1 18
aide 1 o3-mini 18
aide 12 deepseek-r1 46
aide 12 o3-mini 36
aide 125 deepseek-r1 46
aide 125 o3-mini 46
multi_aide 1 deepseek-r1 18
multi_aide 1 o3-mini 18
multi_aide 12 deepseek-r1 18
multi_aide 12 o3-mini 18
multi_aide 125 deepseek-r1 34
multi_aide 125 o3-mini 26


In [7]:
figure_data = {}
for search_algo in search_algo_params.keys():
    for level in [1, 12, 125]:
        for model in ('deepseek-r1', 'o3-mini'):
            plot_info[search_algo][level][model] = process_metrics(plot_info[search_algo][level][model])
            plot_info[search_algo][level][model] = deduplicate_metrics(plot_info[search_algo][level][model])
            figure_data[f'{search_algo}_{level}_{model}'] = convert_to_dict(plot_info[search_algo][level][model])


In [8]:
plot_info['aide'][12]['o3-mini'].keys()

dict_keys(['record_13_20250412_122737_1968005-1968001-2', 'record_14_20250412_122737_1968006-1968001-3', 'record_2_20250412_122724_1967992-1967990-1', 'record_7_20250412_122724_1967997-1967990-6', 'record_11_20250412_122737_1968003-1968001-0', 'record_16_20250412_122737_1968008-1968001-5', 'record_18_20250412_122737_1968001-1968001-7', 'record_5_20250412_122724_1967995-1967990-4', 'record_17_20250412_122737_1968009-1968001-6', 'record_15_20250412_122737_1968007-1968001-4', 'record_10_20250412_122724_1967990-1967990-9', 'record_1_20250412_122724_1967991-1967990-0', 'record_9_20250412_122724_1967999-1967990-8', 'record_12_20250412_122737_1968004-1968001-1', 'record_8_20250412_122724_1967998-1967990-7', 'record_4_20250412_122724_1967994-1967990-3', 'record_3_20250412_122724_1967993-1967990-2', 'record_6_20250412_122724_1967996-1967990-5'])

In [9]:
metrics = plot_info['aide'][12]['o3-mini']['record_15_20250412_122737_1968007-1968001-4']['metrics']

In [10]:
metrics['train_time'].idxmin()

20

In [11]:
# Record	Train Time std
# 1	20183.68
# 2	18300.36
# 3	22270.99
# 4	37980.47
# 5	7365.53
# 6	5922.16
# 7	3805.14
# 8	9496.43
# 9	15747.18
# 10	2315.29
# 11	3259.97
# 12	1220.16
# 13	927.61
# 14	6034.27
# 15	2337.85
# 16	6700.25
# 17	40.5
# 18	1495.18
# 19	600.67
# 20	770.64
# 21	593.64
train_time_std = {
    1: 20183.68,
    2: 18300.36, 
    3: 22270.99,
    4: 37980.47,
    5: 7365.53,
    6: 5922.16,
    7: 3805.14,
    8: 9496.43,
    9: 15747.18,
    10: 2315.29,
    11: 3259.97,
    12: 1220.16,
    13: 927.61,
    14: 6034.27,
    15: 2337.85,
    16: 6700.25,
    17: 40.5,
    18: 1495.18,
    19: 600.67,
    20: 770.64,
    21: 593.64
}

In [13]:

    gaps = {}
    for k, v in human_train_time_dict.items():
        if (k + 1) not in human_train_time_dict:
            continue
        gaps[k] = v - human_train_time_dict[k+1]

In [14]:
std_percentage = {}
for k, v in gaps.items():
    std_percentage[k] = train_time_std[k] / v


{1: 0.026612730115951277,
 2: 0.022215132942209014,
 3: 0.2638524056061701,
 4: 0.1078341169522901,
 5: 0.040189721120320404,
 6: -0.8692440921767209,
 7: 0.034321664697340055,
 8: 0.06061267344932791,
 9: 0.5548493710581023,
 10: 0.06776789111663983,
 11: 0.02604933437744714,
 12: 0.04352429193122637,
 13: 0.055552161935561145,
 14: 0.190692390342561,
 15: 0.27530028261893547,
 16: 0.5318925140906565,
 17: 0.004745722990391376,
 18: 0.12059848362639136}

In [15]:
for key in figure_data.keys():
    figure_data[key] = compute_gap_in_percentage(figure_data[key])
    figure_data[key] = {str(k): v for k, v in figure_data[key].items()}
    del figure_data[key]['6']

In [16]:
std_percentage = {str(k): v for k, v in std_percentage.items()}

In [17]:
# replace the ones with 300% improvement with 0 as they might be summarizer mistakes
for key in figure_data.keys():
    for k, v in figure_data[key].items():
        if v > 3 or v < 0:
            figure_data[key][k] = 0.

In [19]:
# get 6 colors
# colors = ['crimson', 'royalblue', 'forestgreen', 'darkorange', 'purple', 'pink']
# for search_algo in ['flat', 'tree', 'forest', 'aide', 'multi_aide']:
#     filtered_figure_data = {k: v for k, v in figure_data.items() if search_algo in k}
#     data_dicts = []
#     i = 0
#     for model in ('deepseek-r1', 'o3-mini'):
#         for level in [1, 12, 125]:
#             data_dicts.append((
#                 filtered_figure_data[f'{search_algo}_{level}_{model}'], 
#                 f'hints: {level} model: {model}', 
#                 colors[i]
#             ))
#             i += 1
#     plot_gap_comparison(
#         data_dicts, 
#         std_dicts=std_percentage,
#         figsize=(15, 10), 
#         n_cols=3, 
#         main_title=f'Recovered Time Gap Comparison: {search_algo}'
#     )
#     # plt.savefig(f'../figures/{search_algo}_placeholder.pdf', dpi=150)
#     plt.show()



In [28]:
# each model, each level, each search algo, we get a number that is the averaged percetage gap and pass@50%
figure_data.keys()
table_data = {}
for search in ['flat', 'tree', 'forest', 'aide', 'multi_aide']:
    table_data[search] = {}
    for level in [1, 12, 125]:
        table_data[search][level] = {}
        for model in ['deepseek-r1', 'o3-mini']:
            values = figure_data[f'{search}_{level}_{model}'].values()
            avg_percentage = sum(values) / len(values)
            pass_50 = sum(1 for v in values if v >= 0.5)  # count values > 50%
            table_data[search][level][model] = {
                'avg': avg_percentage,
                'pass@50%': pass_50
            }


In [31]:
# Create dataframe for deepseek-r1
deepseek_df_per = pd.DataFrame(
    {search: {level: table_data[search][level]['deepseek-r1']['avg']
             for level in [1, 12, 125]}
     for search in ['flat', 'tree', 'forest', 'aide', 'multi_aide']})

# Create dataframe for o3-mini  
o3_df_per = pd.DataFrame(
    {search: {level: table_data[search][level]['o3-mini']['avg']
             for level in [1, 12, 125]} 
     for search in ['flat', 'tree', 'forest', 'aide', 'multi_aide']})

print("Average time reduction percentage")
print("Deepseek-r1:")
print(deepseek_df_per)
print("\nO3-mini:")
print(o3_df_per)


# Create dataframe for deepseek-r1
deepseek_df_pass = pd.DataFrame(
    {search: {level: f"{table_data[search][level]['deepseek-r1']['pass@50%']}/16"
             for level in [1, 12, 125]}
     for search in ['flat', 'tree', 'forest', 'aide', 'multi_aide']})

# Create dataframe for o3-mini  
o3_df_pass = pd.DataFrame(
    {search: {level: f"{table_data[search][level]['o3-mini']['pass@50%']}/16"
             for level in [1, 12, 125]} 
     for search in ['flat', 'tree', 'forest', 'aide', 'multi_aide']})

print("Pass@50%")
print("Deepseek-r1:")
print(deepseek_df_pass)
print("\nO3-mini:")
print(o3_df_pass)



Average time reduction percentage
Deepseek-r1:
         flat      tree    forest      aide  multi_aide
1    0.432736  0.239397  0.077478  0.153744    0.182068
12   0.285321  0.145057  0.485767  0.389881    0.400670
125  0.374141  0.420214  0.522341  0.370704    0.493392

O3-mini:
         flat      tree    forest      aide  multi_aide
1    0.437768  0.414583  0.347951  0.437201    0.326087
12   0.469600  0.406840  0.451358  0.501255    0.445816
125  0.638741  0.677204  0.569546  0.594289    0.539397
Pass@50%
Deepseek-r1:
     flat  tree forest  aide multi_aide
1    7/18  2/18   1/18  2/18       3/18
12   3/18  3/18   7/18  5/18       5/18
125  5/18  6/18   8/18  5/18       7/18

O3-mini:
     flat   tree forest  aide multi_aide
1    6/18   7/18   6/18  8/18       5/18
12   7/18   6/18   6/18  7/18       7/18
125  9/18  11/18   8/18  9/18       8/18
